In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
import implicit

In [2]:
user_item_itera = pd.read_parquet("../data/output/user-item-interaction.parquet")
user_item_itera

,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,...,num_item_img,num_item_videos,is_free,has_price_info,price_bucket,num_categories,days_since_review,review_year,review_month,recency_weight
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,...,21,10,False,True,25-50,3,1531,2019,7,0.015121
1,1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,lots of fun,i love playing tapped out because it is fun to...,[],True,0,...,6,0,True,True,unknown,0,3129,2015,2,0.000190
2,2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,light up the dark,i love this flashlight app! it really illumin...,[],True,0,...,4,0,True,True,unknown,0,3843,2013,3,0.000027
3,3,b00kcymawk,b00kcymawk,ah6catodivpvuojewhrsrcskaoha,2019-06-20,fun game,one of my favorite games,[],True,0,...,8,1,True,True,unknown,0,1544,2019,6,0.014593
4,4,b00p1rk566,b00p1rk566,aeiny4xoinmmjck5gz3m6mmhbn6a,2014-12-11,i am not that good at it but my kids are,cute game. i am not that good at it but my kid...,[],True,0,...,6,0,False,True,0-1,0,3196,2014,12,0.000158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4880176,4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,gog,very fun and addictive and exciting,[],True,0,...,10,1,True,True,unknown,0,990,2020,12,0.066505
4880177,4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,worst game ever,worst game ever toxic people and bad connectio...,[],True,1,...,6,0,True,True,unknown,0,978,2021,1,0.068727
4880178,4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,better!!!,this fabulous game is 10000 times better than ...,[],True,2,...,9,0,False,True,1-10,0,3758,2013,5,0.000034
4880179,4880179,b00m9j3ioa,b00m9j3ioa,ahqetdykhvdngmgbwvjl6vjxjgfq,2016-09-03,it has everything i need and more,awesome! i upgraded from coreldraw 8. i was wo...,[],True,0,...,24,2,False,False,NaN,3,2564,2016,9,0.000894


In [3]:
user_item_itera.columns

Index(['review_id', 'asin', 'parent_asin', 'user_id', 'review_datetime',
       'review_title', 'review_text', 'review_images', 'verified_purchase',
       'helpful_vote', 'review_rating', 'review_text_length',
       'review_title_length', 'review_word_count', 'is_extreme_rating',
       'is_positive', 'is_negative', 'num_review_img', 'item_title',
       'main_category', 'categories', 'description', 'features', 'details',
       'item_images', 'item_videos', 'item_rating', 'store', 'price',
       'num_item_img', 'num_item_videos', 'is_free', 'has_price_info',
       'price_bucket', 'num_categories', 'days_since_review', 'review_year',
       'review_month', 'recency_weight'],
      dtype='str')

In [6]:
# Keep only relevant columns
df = user_item_itera[['user_id','asin','review_rating']].copy()
df = df.groupby(['user_id','asin'], as_index=False)['review_rating'].mean()

# Create categorical codes
df['user_idx'] = df['user_id'].astype('category').cat.codes
df['item_idx'] = df['asin'].astype('category').cat.codes

# Interaction strength
df['interaction'] = df['review_rating'].astype(float)

# Mapping dictionaries
user_to_idx = dict(zip(df['user_id'], df['user_idx']))
idx_to_user = dict(zip(df['user_idx'], df['user_id']))
item_to_idx = dict(zip(df['asin'], df['item_idx']))
idx_to_item = dict(zip(df['item_idx'], df['asin']))

# Sparse matrix
user_item_matrix = csr_matrix((df['interaction'], (df['user_idx'], df['item_idx'])))
user_item_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4829120 stored elements and shape (2589466, 90359)>

In [7]:
# Train ALS
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20)
model.fit(user_item_matrix.T)  # transpose: ALS expects item-user

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.09309768676757812 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

In [8]:
# Get the *categorical code* for the user
some_user_id = df['user_id'].iloc[0]
user_code = user_to_idx[some_user_id]  # this must be between 0 and num_users-1

# Recommend
recomm_asins, recomm_scores = model.recommend(
    userid=user_code,
    user_items=user_item_matrix[user_code],
    N=5
)

recomm_asins

array([1819021,  217143, 1020914,  359435, 1297237], dtype=int32)

### Save Model

In [ ]:
# import pickle

# with open("cf_model.pkl", "wb") as f:
#     pickle.dump(model, f)

# with open("user_item_matrix.pkl", "wb") as f:
#     pickle.dump(user_item_matrix, f)

### Load model for inference tool

In [ ]:
# with open("cf_model.pkl", "rb") as f:
#     model = pickle.load(f)

# with open("user_item_matrix.pkl", "rb") as f:
#     user_item_matrix = pickle.load(f)